# Lean-15 — Dérivées symboliques de Brzozowski : la finitude qui garantit le *matching* linéaire

**Série SymbolicAI/Lean** — Pont vers l'Epic #2978 (*le Sudoku comme regex symbolique*), livrable **C (Lean)**. See #2978 (partial), See #2162 (homonymie Conway).

> *Note d'homonymie posée d'entrée.* Le héros de notre Epic Lean `conway_lean` (#2162) est **John Horton Conway**, mathématicien, créateur du *Game of Life*. Le célèbre *sudoku en regex Perl* (Perlmonks, 2002) est l'œuvre de **Damian Conway**, linguiste-informaticien. **Deux Conway distincts.** Les deux se « rencontrent » dans cette série : l'un par la reconnaissance par regex, l'autre par la formalisation du Jeu de la Vie. Ce notebook documente ce clin d'œil, pas une confusion.

## 1. Le problème : pourquoi un reconnaisseur regex *devrait* être linéaire

Un reconnaisseur d'expressions régulières (*regex matcher*) lit un mot caractère par caractère. Deux familles :

- **Backtracking** (PCRE, moteurs « NFA traditionnels ») : en cas d'échec, le moteur *revient en arrière* et essaie une autre branche. Simple à implémenter, mais la complexité peut devenir **exponentielle** (catastrophic backtracking). C'est la famille du *sudoku en regex* de Damian Conway — un monstre de lookaheads gigantesques.
- **Non-backtracking** (DFA / dérivées) : on consomme **chaque caractère une seule fois**, sans retour. Complexité **linéaire** en la longueur du mot. C'est la famille de `.NET RegexOptions.NonBacktracking` (PLDI 2023), de **RE#** (POPL 2025) et de SymPy.

La question de fond : **comment garantir qu'un reconnaisseur non-backtracking termine ?** Réponse : si l'ensemble des « états » qu'il peut visiter est **fini**. Et cet ensemble d'états, c'est exactement l'ensemble des **dérivées** de la regex — un concept dû à Janusz Brzozowski en 1964.

## 2. La dérivée de Brzozowski (1964)

Soit une regex $r$ et un caractère $a$. La **dérivée** $D_a(r)$ est la regex qui reconnaît exactement les mots $w$ tels que le mot $a{\cdot}w$ est reconnu par $r$ :

$$L(D_a(r)) = \{\, w \mid a{\cdot}w \in L(r) \,\}$$

Intuition : *« j'ai lu $a$, que me reste-t-il à reconnaître ? »* La définition est récursive sur la structure de $r$ :

| $r$ | $D_a(r)$ |
|-----|----------|
| $\emptyset$ | $\emptyset$ |
| $\varepsilon$ | $\emptyset$ |
| $b$ (un caractère) | $\varepsilon$ si $a = b$, sinon $\emptyset$ |
| $r{\cdot}s$ | $D_a(r){\cdot}s \;\cup\; (\varepsilon \in L(r) ? D_a(s) : \emptyset)$ |
| $r \cup s$ | $D_a(r) \cup D_a(s)$ |
| $r^*$ | $D_a(r){\cdot}r^*$ |

Le *matching* non-backtracking s'écrit alors trivialement : un mot $w = a_1 a_2 \dots a_n$ est reconnu par $r$ **ssi** la dérivée itérée $D_{a_n}(\dots D_{a_1}(r)\dots)$ est **nullable** (reconnaît le mot vide $\varepsilon$). On lit chaque caractère une seule fois, en avant.

## 3. Le théorème de finitude — *le* résultat qui garantit la terminaison

> **Théorème (Brzozowski, 1964).** L'ensemble des dérivées itérées $\{D_w(r) \mid w \in \Sigma^*\}$ d'une expression régulière $r$ est **fini** — modulo l'équivalence ACI (Associativité, Commutativité, Idempotence) des unions.

C'est ce théorème qui transforme la dérivée d'un concept élégant en un **algorithme qui termine** : comme l'espace des dérivées est fini, un reconnaisseur qui passe d'une dérivée à la suivante ne peut visiter qu'un nombre borné d'états distincts. D'où la complexité **linéaire** des moteurs RE# et `.NonBacktracking`.

Ci-dessous, nous illustrons ce concept sur un **mini-projet Lean autonome** (sans dépendance Mathlib).

In [1]:
import subprocess
from pathlib import Path

# Le mini-projet Lake no-Mathlib (autonome, a cote des autres projets Lean de la serie)
LEAN_PROJECT = '/mnt/c/dev/CoursIA-2/MyIA.AI.Notebooks/SymbolicAI/Lean/finiteness_lean'

def wsl(cmd, timeout=180):
    """Execute une commande bash dans WSL Ubuntu."""
    full = ['wsl', '-d', 'Ubuntu', '--', 'bash', '-lc', cmd]
    try:
        r = subprocess.run(full, capture_output=True, text=True, timeout=timeout)
        return r.returncode, r.stdout, r.stderr
    except subprocess.TimeoutExpired:
        return -1, '', f'TIMEOUT after {timeout}s'

rc, out, err = wsl(f'cat {LEAN_PROJECT}/lean-toolchain')
print('Toolchain:', out.strip())
print('WSL bridge: OK')

Toolchain: leanprover/lean4:v4.30.0-rc2
WSL bridge: OK


In [2]:
# Construction du mini-projet Lean (no-Mathlib : rapide, ~10s)
rc, out, err = wsl(f'cd {LEAN_PROJECT} && lake build Finiteness 2>&1')
print(out)
if rc != 0:
    print('STDERR:', err)
print(f'\n[exit={rc}]  Build complet du projet Finiteness (inductive Regex + deriv + exemples).')

ℹ [2/4] Replayed Finiteness.Basic
info: Finiteness/Basic.lean:99:0: true
info: Finiteness/Basic.lean:100:0: false
info: Finiteness/Basic.lean:106:0: true
info: Finiteness/Basic.lean:107:0: false
info: Finiteness/Basic.lean:108:0: false
Build completed successfully (4 jobs).


[exit=0]  Build complet du projet Finiteness (inductive Regex + deriv + exemples).


In [3]:
# Le source Lean original (concept public de Brzozowski 1964 -- PAS le code d'ezhuchko)
rc, out, err = wsl(f'cat {LEAN_PROJECT}/Finiteness/Basic.lean')
print(out)

/-! # Dérivées symboliques de Brzozowski — démonstration pédagogique autonome

Soit une expression régulière `r` sur un alphabet `α`. La **dérivée** de `r`
par un caractère `a` (Janusz Brzozowski, *Derivatives of Regular Expressions*,
JACM 1964) est l'expression `D_a(r)` qui reconnaît exactement les mots `w` tels
que le mot `a :: w` est reconnu par `r`. Itérée sur les caractères d'un mot, la
dérivée calcule la correspondance (*matching*) **sans reculer**
(non-backtracking) : un mot `w` est reconnu par `r` si et seulement si la
dérivée de `r` par `w` est *nullable* (reconnaît le mot vide ε).

## Le théorème de finitude

Brzozowski (1964) démontre que l'ensemble des dérivées itérées d'une expression
régulière est **fini** — modulo l'équivalence ACI (Associativité, Commutativité,
Idempotence) des unions. C'est cette finitude qui garantit la **terminaison** et
la complexité **linéaire** des reconnaisseurs modernes :
  * .NET `RegexOptions.NonBacktracking` (PLDI 2023),
  * **RE#** (POPL 202

### Lecture des sorties

Les `#eval` du fichier compilent et s'exécutent (capturées dans la sortie `lake build` ci-dessus) :

- `accepts ['a','a','a','a'] aStar` → **`true`** : le mot "aaaa" appartient a `a*` (l'étoile reconnaît toute répétition de 'a').
- `accepts ['a','b'] aStar` → **`false`** : "ab" n'appartient pas a `a*` (le 'b' casse la répétition).
- `accepts ['a','b'] abWord` → **`true`**, `accepts ['a'] abWord` → **`false`**, `accepts ['a','b','c'] abWord` → **`false`** : le mot "ab" et lui seul.

Ces résultats sont obtenus par **dérivation itérée + test de nullabilité** — la définition même du *matching* non-backtracking. Les deux `example` (`deriv 'a' (char 'a') = eps`, `deriv 'b' (char 'a') = empty`) sont **prouvés** par `simp [deriv]` : on consomme un caractère singleton, ou on échoue.

## 4. Le point fixe — pourquoi l'espace des dérivées est minuscule

Considérez `r = a*` (l'étoile sur le caractère 'a'). Calculons sa dérivée par 'a' :

$$D_a(a^*) = D_a(a){\cdot}a^* = \varepsilon{\cdot}a^* \equiv a^*$$

**La dérivée de `a*` par 'a' est `a*` elle-même** : c'est un *point fixe*. L'espace des dérivées de `a*` est donc réduit au **singleton {a*}**. Le reconnaisseur n'a qu'un seul état a visiter — d'où une reconnaissance en temps strictement linéaire, sans aucune structure de données auxiliaire.

C'est cette propriété — *finitude de l'espace des dérivées* — que Brzozowski (1964) a démontrée en général, et qui sous-tend tous les moteurs linéaires modernes.

## 5. La formalisation Lean constructive — Zhuchko, Maarand, Veanes, Ebner (ITP 2025)

La formalisation **complète** en Lean 4 du théorème de finitude est due a **Ekaterina Zhuchko, Hendrik Maarand, Margus Veanes, Gabriel Ebner** :

> E. Zhuchko, H. Maarand, M. Veanes, G. Ebner. *Finiteness of Symbolic Derivatives in Lean*. ITP 2025. Dépôt : [`ezhuchko/finiteness-derivatives`](https://github.com/ezhuchko/finiteness-derivatives).

La preuve est **constructive** : étant donnée une expression $R$, on **construit** un ensemble fini qui surestime (modulo l'équivalence des dérivées) l'ensemble des dérivées de $R$. Elle réutilise l'infrastructure de formalisation des expressions régulières en Lean 4, illustrant la réutilisabilité du cadre.

> ⚠️ **Licence amont non confirmée.** Conformément a l'issue #2978, ce notebook **ne vendore pas** le code d'ezhuchko. Le mini-projet `finiteness_lean/` ci-dessus est **entièrement original** : il illustre le concept public de Brzozowski (1964) par des définitions et exemples écrits spécifiquement pour cette série pédagogique, sans aucune dépendance (Mathlib exclu). Toute réutilisation du dépôt `ezhuchko/finiteness-derivatives` est subordonnée a la confirmation de sa licence.

**Companion** : CPP 2024 (extended regex matching with lookarounds, Zhuchko/Veanes/Ebner) — même lignée de recherche.

## 6. Le pont vers l'Epic Conway #2162

Ce notebook est le **troisième pilier** du triptyque #2978 :

| Pilier | Outil | Rôle |
|--------|-------|------|
| **Résoudre** (produire une grille) | Z3 (lignée Rex / `RegexToSMT`) | Le résolveur — génère un témoin |
| **Vérifier** (valider une grille) | RE# / `.NonBacktracking` | Le vérificateur — reconnaissance en temps linéaire |
| **Prouver** (garantir la terminaison) | Lean — dérivées de Brzozowski | Ce notebook — la finitude sous-jacente |

Les trois opérations distinctes du paysage regex symbolique : **résoudre ≠ vérifier ≠ prouver**. La reconnaissance non-backtracking de RE# *est* la finitude des dérivées, rendue algorithmique.

Et le clin d'œil final : le héros de notre Epic Lean `conway_lean` — la formalisation du *Game of Life* de **John Horton Conway** — « croise » ici le *sudoku en regex* de **Damian Conway**. Deux Conway, deux génies, une même série SymbolicAI/Lean.

Voir [`conway_lean`](conway_lean/) (Epic #2162, preuve de correction Hashlife) et le notebook [`Lean-14-Conway-Tribute`](Lean-14-Conway-Tribute.ipynb).

## Exercices

Les exercices suivants se complètent dans le fichier `Finiteness/Basic.lean` du mini-projet, puis se valident par `lake build Finiteness` (cellule ci-dessus).

**Exercice 1 (dérivée).** Ajoutez un constructeur `Regex.plus : Regex α → Regex α` (le langage $r^+$ = une occurrence ou plus). Définissez son cas dans `nullable` et dans `deriv`. *Indice : $r^+ = r{\cdot}r^*$, donc $D_a(r^+) = D_a(r){\cdot}r^*$.*

```lean
-- TODO étudiant : constructor + nullable case + deriv case
-- def nullable ... | plus r => ...
-- def deriv   ... | plus r => ...
```

**Exercice 2 (reconnaissance).** Construisez la regex du langage « les mots sur {a,b} contenant au moins un 'a' », puis vérifiez par `#eval accepts` que "ba" est reconnu mais "bb" ne l'est pas. *Indice : `(a∪b)* · a · (a∪b)*`.*

```lean
-- TODO étudiant
-- def hasA : Regex Char := ...
-- #eval accepts ['b','a'] hasA   -- attendu : true
-- #eval accepts ['b','b'] hasA   -- attendu : false
```

**Exercice 3 (finitude explicite).** Montrez par le calcul (une série de `#eval` de `deriv` itérés) que l'espace des dérivées de `concat (char 'a') (char 'b')` est bien fini : partez de `abWord`, dérivez par 'a', puis par 'b', et observez que la dérivée finale est nullable (`eps`) puis devient `empty` si on dérive encore. *Indice : `#eval deriv 'a' abWord`, `#eval deriv 'b' (deriv 'a' abWord)`.*